# KV-cache calculator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/calculators/kv-cache.ipynb)

How much GPU memory does a model need just to *remember* the conversation? While generating, a transformer keeps the keys and values of every previous token for every layer. See [Attention](../03-foundations/attention.md) for why.

$$\text{bytes per token} = 2 \times \text{layers} \times \text{KV heads} \times \text{head dim} \times \text{bytes per element}$$

Runs on CPU. Change the numbers and re-run.

In [ ]:
# Use the repo's tested package; install it from GitHub if needed (e.g. on Colab).
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

from s2a.kv_cache import ModelConfig, LLAMA3_8B, bytes_per_token, kv_cache_bytes, max_tokens_in_memory, human_bytes

## 1. One model, one conversation

In [ ]:
model = LLAMA3_8B          # 32 layers, 8 KV heads (GQA), head_dim 128
seq_len = 8192
dtype = "bf16"

print(f"Per token: {human_bytes(bytes_per_token(model, dtype))}")
print(f"{seq_len:,} tokens: {human_bytes(kv_cache_bytes(model, seq_len, dtype=dtype))}")

## 2. Why GQA exists: MHA vs GQA vs MQA

In [ ]:
for name, kv_heads in [("MHA", 32), ("GQA", 8), ("MQA", 1)]:
    m = ModelConfig(name, layers=32, kv_heads=kv_heads, head_dim=128)
    print(f"{name}: {human_bytes(bytes_per_token(m)):>10} per token, "
          f"{human_bytes(kv_cache_bytes(m, 8192)):>9} for 8k tokens")

## 3. How many users fit on a GPU?

Memory left for the KV cache is roughly GPU memory minus the weights (and some overhead). An 8B-parameter model in BF16 is about 16 GB of weights.

In [ ]:
GB = 1e9
gpu_memory = 80 * GB       # e.g. an 80 GB GPU
weights = 16 * GB
overhead = 4 * GB          # activations, fragmentation, runtime
free = gpu_memory - weights - overhead

for dtype in ["bf16", "fp8"]:
    tokens = max_tokens_in_memory(LLAMA3_8B, free, dtype)
    print(f"{dtype}: {tokens:,} tokens -> {tokens // 8192} concurrent 8k-token conversations")

## Try it

- Look up a model's config (`num_hidden_layers`, `num_key_value_heads`, `head_dim`) on Hugging Face and plug it in.
- What context length uses up the whole GPU for a single user?
- This formula doesn't fit MLA or hybrid linear-attention models. Why not?